# FUSE Stage 3 — partial alignment, prior adaptation, and missing-piece hypothesis

This notebook consumes two upstream container results:

- **VGGT measured geometry** (authoritative): `data/vggt_outputs/global/broken_clean_normals.ply`
- **Hunyuan intact prior** (hypothesis): the newest `data/hunyuan_outputs/runs/*/selected/prior_manifest.json`
  and its `intact_prior.glb`

It performs:

1. input discovery and contract validation;
2. partial Sim(3) initialization (manual landmarks when supplied, otherwise PCA hypotheses);
3. robust trimmed partial ICP;
4. Kaolin differentiable overlap refinement;
5. optional object-agnostic embedded-deformation-graph adaptation;
6. supported / uncertain / unsupported surface classification;
7. user-confirmed missing-region extraction and validation.

**Authority rule:** the VGGT surface is never deformed or averaged with the prior. Only the Hunyuan
prior moves. An unsupported prior surface is not automatically called “missing”: it may be absent
from VGGT because it was occluded. Final extraction therefore requires explicit user confirmation
of one or more candidate components.


## 0. Environment and reproducibility

The original starter notebook was executed with Torch 2.5.1, CUDA 12.4 and Kaolin 0.18.0.
This version also requires `scipy` and `trimesh`. If the import cell reports that `trimesh` is
missing, add `trimesh>=4.4,<5` to the Kaolin container requirements and rebuild it.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import permutations, product
import hashlib
import json
import math
import os
import platform
import warnings
import shutil

import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation
import torch
import kaolin as kal
import open3d as o3d
import trimesh
import plotly.graph_objects as go
from IPython.display import display

SEED = 17
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
print("kaolin:", kal.__version__)
print("open3d:", o3d.__version__)
print("trimesh:", trimesh.__version__)
print("device:", DEVICE)


## 1. Configuration

Normally only three edits are needed:

1. set explicit input overrides if automatic discovery selects the wrong run;
2. add 5–8 corresponding landmarks if PCA initialization is ambiguous;
3. after candidate inspection, set `CONFIRM_OBSERVED_ABSENCE=True` and select component IDs.


In [ ]:
FUSE_ROOT = Path("/workspace")
DATA_DIR = FUSE_ROOT / "data" 
SCENE = "global"

# Optional absolute paths. Leave as None for contract-based discovery.
VGGT_CLOUD_OVERRIDE = None
HUNYUAN_PRIOR_OVERRIDE = None
HUNYUAN_MANIFEST_OVERRIDE = None

RUN_NAME = os.environ.get("FUSE_ALIGNMENT_RUN", "current")
OUT_DIR = DATA_DIR / "kaolin_outputs" / "alignment" / "runs" / RUN_NAME
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Coarse registration.
COARSE_SAMPLE_PRIOR = 20_000
COARSE_SAMPLE_VGGT = 30_000
PCA_SCORE_SAMPLE = 8_000
PARTIAL_ICP_ITERATIONS = 25
PARTIAL_ICP_TRIM_QUANTILE = 0.68
PARTIAL_ICP_NORMAL_COS = math.cos(math.radians(70.0))
SIM3_SCALE_WINDOW = (0.72, 1.38)  # relative to the initializer
PCA_HYPOTHESIS_RANK = 0            # try 1, 2, ... only if the rank-0 overlay has the wrong orientation

# Kaolin rigid refinement. Conservative sizes for an RTX 4080 Laptop GPU.
KAOLIN_PRIOR_POINTS = 14_000
KAOLIN_VGGT_POINTS = 24_000
KAOLIN_RIGID_STEPS = 350
KAOLIN_RIGID_LR = 2e-3
KAOLIN_TRIM_QUANTILE = 0.68
KAOLIN_REVERSE_WEIGHT = 0.15
KAOLIN_NORMAL_WEIGHT = 0.10
KAOLIN_LANDMARK_WEIGHT = 1.0
KAOLIN_REG_WEIGHT = 0.01

# Generic embedded deformation graph.
RUN_GENERIC_DEFORMATION = True
DEFORM_GRAPH_NODES = 96
DEFORM_GRAPH_KNN = 4
DEFORM_STEPS = 300
DEFORM_LR = 3e-3
DEFORM_FIT_REL_DIAGONAL = 0.05
DEFORM_MAX_TRANSLATION_REL = 0.06
DEFORM_MAX_ROTATION_DEG = 25.0
DEFORM_W_NORMAL = 0.08
DEFORM_W_ARAP = 5.0
DEFORM_W_ANCHOR = 8.0
DEFORM_W_ROTATION = 0.02
DEFORM_W_TRANSLATION = 0.05

# Support classification.
SUPPORT_REL_DIAGONAL = 0.008
SUPPORT_SPACING_MULTIPLIER = 4.0
SUPPORT_NORMAL_COS = math.cos(math.radians(65.0))
UNCERTAIN_MULTIPLIER = 2.5
MIN_COMPONENT_FACES = 40
MAX_COMPONENTS_TO_SHOW = 12

# Missing-region confirmation. Fill these only after running the candidate preview cell.
SELECT_MISSING_COMPONENTS = []       # e.g. [0]
MISSING_SEEDS = np.empty((0, 3))     # optional points in the aligned VGGT frame
CONFIRM_OBSERVED_ABSENCE = False     # deliberately False on a first run
ATTACHMENT_MODE = "exterior_patch"  # "exterior_patch" or best-effort "solid_cap"

print("output directory:", OUT_DIR)


## 2. Upstream contract discovery

VGGT discovery prefers the normals-bearing cleaned cloud and falls back to the original starter
path. Hunyuan discovery prefers the newest selected manifest, reads its `main_output`, and falls
back to a selected `intact_prior.glb`. Explicit overrides always win.


In [ ]:
def _as_workspace_path(value):
    if value is None:
        return None
    path = Path(value)
    if path.is_absolute():
        return path
    return FUSE_ROOT / path


def _newest(paths):
    paths = [Path(p) for p in paths if Path(p).exists()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None


def discover_vggt_cloud():
    if VGGT_CLOUD_OVERRIDE:
        return _as_workspace_path(VGGT_CLOUD_OVERRIDE)

    preferred = [
        # DATA_DIR / "cleaned_geometry" / SCENE / "broken_clean_normals.ply",
        # DATA_DIR / "cleaned_geometry" / SCENE / "broken_clean.ply",
        DATA_DIR / "vggt_outputs" / SCENE / "broken_clean_normals.ply",
        DATA_DIR / "vggt_outputs" / SCENE / "broken_clean.ply",
        DATA_DIR / "vggt_outputs" / SCENE / "cleaned_cloud.ply",
        DATA_DIR / "vggt_outputs" / SCENE / "raw_vggt_cloud.ply",
    ]
    for path in preferred:
        if path.exists():
            return path

    matches = list(DATA_DIR.glob("**/broken_clean_normals.ply"))
    matches += list(DATA_DIR.glob("**/broken_clean.ply"))
    return _newest(matches)


def discover_hunyuan_manifest():
    if HUNYUAN_MANIFEST_OVERRIDE:
        return _as_workspace_path(HUNYUAN_MANIFEST_OVERRIDE)
    return _newest(DATA_DIR.glob("hunyuan_outputs/runs/*/selected/prior_manifest.json"))


def discover_hunyuan_prior(manifest_path=None):
    if HUNYUAN_PRIOR_OVERRIDE:
        return _as_workspace_path(HUNYUAN_PRIOR_OVERRIDE)

    if manifest_path and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        for key in ("main_output", "alignment_glb", "intact_prior"):
            value = manifest.get(key)
            if value:
                candidate = _as_workspace_path(value)
                if candidate.exists():
                    return candidate

    matches = list(DATA_DIR.glob("hunyuan_outputs/runs/*/selected/intact_prior.glb"))
    return _newest(matches)


VGGT_CLOUD_PATH = discover_vggt_cloud()
HUNYUAN_MANIFEST_PATH = discover_hunyuan_manifest()
HUNYUAN_PRIOR_PATH = discover_hunyuan_prior(HUNYUAN_MANIFEST_PATH)

contract = {
    "vggt_cloud": str(VGGT_CLOUD_PATH) if VGGT_CLOUD_PATH else None,
    "hunyuan_manifest": str(HUNYUAN_MANIFEST_PATH) if HUNYUAN_MANIFEST_PATH else None,
    "hunyuan_prior": str(HUNYUAN_PRIOR_PATH) if HUNYUAN_PRIOR_PATH else None,
}
print(json.dumps(contract, indent=2))

if VGGT_CLOUD_PATH is None or not VGGT_CLOUD_PATH.exists():
    raise FileNotFoundError("No VGGT cloud matched the Stage 1 contract. Set VGGT_CLOUD_OVERRIDE.")
if HUNYUAN_PRIOR_PATH is None or not HUNYUAN_PRIOR_PATH.exists():
    raise FileNotFoundError("No selected Hunyuan prior matched the Stage 2 contract. Set HUNYUAN_PRIOR_OVERRIDE.")

(OUT_DIR / "input_contract.json").write_text(json.dumps(contract, indent=2))


In [ ]:
def file_sha256(path, block_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def load_trimesh(path):
    loaded = trimesh.load(path, force="scene", process=False)
    if isinstance(loaded, trimesh.Scene):
        if not loaded.geometry:
            raise ValueError(f"No geometry in {path}")
        try:
            mesh = loaded.to_mesh() ## mesh = loaded.dump(concatenate=True)
        except Exception:
            mesh = trimesh.util.concatenate(tuple(loaded.geometry.values()))
    else:
        mesh = loaded
    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a triangle mesh, got {type(mesh).__name__}")
    mesh = mesh.copy()
    finite = np.isfinite(mesh.vertices).all(axis=1)
    if not finite.all():
        mesh.update_vertices(finite)
    mesh.remove_unreferenced_vertices()
    if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise ValueError(f"Empty mesh after loading {path}")
    return mesh


def load_vggt_cloud(path):
    cloud = o3d.io.read_point_cloud(str(path))
    if len(cloud.points) == 0:
        raise ValueError(f"Empty point cloud: {path}")
    points = np.asarray(cloud.points, dtype=np.float64)
    finite = np.isfinite(points).all(axis=1)
    if not finite.all():
        cloud = cloud.select_by_index(np.flatnonzero(finite))
        points = np.asarray(cloud.points, dtype=np.float64)

    diagonal = float(np.linalg.norm(np.ptp(points, axis=0)))
    if not cloud.has_normals():
        radius = max(diagonal * 0.025, 1e-6)
        cloud.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=60)
        )
        try:
            cloud.orient_normals_consistent_tangent_plane(30)
        except RuntimeError:
            warnings.warn("Could not orient VGGT normals consistently; absolute normal cosine will be used.")

    points = np.asarray(cloud.points, dtype=np.float64)
    normals = np.asarray(cloud.normals, dtype=np.float64)
    colors = (
        np.asarray(cloud.colors, dtype=np.float64)
        if cloud.has_colors()
        else np.full_like(points, 0.65)
    )
    return cloud, points, normals, colors


vggt_cloud, vggt_points, vggt_normals, vggt_colors = load_vggt_cloud(VGGT_CLOUD_PATH)
prior_mesh = load_trimesh(HUNYUAN_PRIOR_PATH)

VGGT_DIAGONAL = float(np.linalg.norm(np.ptp(vggt_points, axis=0)))
if not np.isfinite(VGGT_DIAGONAL) or VGGT_DIAGONAL <= 0:
    raise ValueError("VGGT cloud has an invalid bounding-box diagonal.")

print("VGGT points:", len(vggt_points))
print("VGGT bbox min/max:", vggt_points.min(0), vggt_points.max(0))
print("VGGT diagonal:", VGGT_DIAGONAL)
print("Hunyuan vertices/faces:", len(prior_mesh.vertices), len(prior_mesh.faces))
print("Hunyuan extents:", prior_mesh.extents)
print("input hashes:")
print("  VGGT:", file_sha256(VGGT_CLOUD_PATH))
print("  prior:", file_sha256(HUNYUAN_PRIOR_PATH))


## 3. Input inspection

At this point the meshes are intentionally shown in their native frames. They are not expected to
overlap yet. Hovering over points reveals coordinates that can be copied into the landmark cell.


In [ ]:
def sample_rows(array, count, rng):
    array = np.asarray(array)
    if len(array) <= count:
        return array.copy()
    return array[rng.choice(len(array), count, replace=False)]


def sample_surface_with_normals(mesh, count, seed=SEED):
    state = np.random.get_state()
    np.random.seed(seed)
    try:
        points, face_index = trimesh.sample.sample_surface(mesh, count)
    finally:
        np.random.set_state(state)
    normals = np.asarray(mesh.face_normals)[face_index]
    return np.asarray(points), np.asarray(normals)


def overlay_figure(measured, prior=None, prior_colors=None, title="FUSE geometry overlay", max_points=70_000):
    rng = np.random.default_rng(SEED)
    measured_show = sample_rows(measured, max_points, rng)
    traces = [
        go.Scatter3d(
            x=measured_show[:, 0], y=measured_show[:, 1], z=measured_show[:, 2],
            mode="markers", name="VGGT measured",
            marker=dict(size=1.8, color="rgb(120,125,130)", opacity=0.75),
            hovertemplate="VGGT<br>x=%{x:.6f}<br>y=%{y:.6f}<br>z=%{z:.6f}<extra></extra>",
        )
    ]
    if prior is not None:
        prior = np.asarray(prior)
        if len(prior) > max_points:
            idx = rng.choice(len(prior), max_points, replace=False)
            prior_show = prior[idx]
            colors_show = None if prior_colors is None else np.asarray(prior_colors)[idx]
        else:
            prior_show = prior
            colors_show = prior_colors
        traces.append(
            go.Scatter3d(
                x=prior_show[:, 0], y=prior_show[:, 1], z=prior_show[:, 2],
                mode="markers", name="intact prior",
                marker=dict(
                    size=2.0,
                    color="rgb(20,190,220)" if colors_show is None else colors_show,
                    opacity=0.55,
                ),
                hovertemplate="prior<br>x=%{x:.6f}<br>y=%{y:.6f}<br>z=%{z:.6f}<extra></extra>",
            )
        )
    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        scene=dict(aspectmode="data"),
        width=1000, height=760,
        margin=dict(l=0, r=0, b=0, t=45),
        legend=dict(x=0.01, y=0.99),
    )
    return fig


prior_preview_points, _ = sample_surface_with_normals(prior_mesh, 60_000)
fig = overlay_figure(vggt_points, prior_preview_points, title="Native frames — alignment not yet applied")
fig.show()
fig.write_html(OUT_DIR / "00_native_frames.html", include_plotlyjs=True, full_html=True)


## 4. Coarse partial Sim(3)

A similarity transform is (T(x)=sRx+t). If you can identify the same points on the surviving
object and on the intact prior, enter 5–8 corresponding 3D landmarks below. They must be in the
same order. Do not pick the inferred missing region or the measured fracture face.

If the arrays remain empty, the notebook evaluates the 24 proper PCA-axis hypotheses and refines
the best one with robust trimmed partial ICP. PCA initialization is object-independent, but it can
be ambiguous for symmetric objects. Always inspect the resulting overlay.


In [ ]:
# Replace these empty arrays with corresponding coordinates when automatic initialization is wrong.
LANDMARKS_PRIOR = np.empty((0, 3), dtype=np.float64)
LANDMARKS_VGGT = np.empty((0, 3), dtype=np.float64)

# Example only (delete the leading # and replace the numbers):
# LANDMARKS_PRIOR = np.array([[x1, y1, z1], [x2, y2, z2], [x3, y3, z3], [x4, y4, z4]])
# LANDMARKS_VGGT  = np.array([[X1, Y1, Z1], [X2, Y2, Z2], [X3, Y3, Z3], [X4, Y4, Z4]])

if LANDMARKS_PRIOR.shape != LANDMARKS_VGGT.shape or LANDMARKS_PRIOR.ndim != 2 or LANDMARKS_PRIOR.shape[1] != 3:
    raise ValueError("LANDMARKS_PRIOR and LANDMARKS_VGGT must both have shape (N, 3).")
if 0 < len(LANDMARKS_PRIOR) < 3:
    raise ValueError("Use at least three non-collinear landmark pairs, preferably 5–8.")

print("manual landmark pairs:", len(LANDMARKS_PRIOR))


In [ ]:
def transform_points(points, scale, rotation, translation):
    return scale * np.asarray(points) @ np.asarray(rotation).T + np.asarray(translation)


def transform_normals(normals, rotation):
    transformed = np.asarray(normals) @ np.asarray(rotation).T
    return transformed / np.maximum(np.linalg.norm(transformed, axis=1, keepdims=True), 1e-12)


def sim3_matrix(scale, rotation, translation):
    matrix = np.eye(4, dtype=np.float64)
    matrix[:3, :3] = scale * np.asarray(rotation)
    matrix[:3, 3] = np.asarray(translation)
    return matrix


def weighted_umeyama(source, target, weights=None):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    if source.shape != target.shape or source.ndim != 2 or source.shape[1] != 3:
        raise ValueError("source and target must both have shape (N, 3)")
    if len(source) < 3:
        raise ValueError("At least three correspondences are required")

    if weights is None:
        weights = np.ones(len(source), dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.maximum(weights, 0)
    weights /= max(weights.sum(), 1e-12)

    source_mean = np.sum(source * weights[:, None], axis=0)
    target_mean = np.sum(target * weights[:, None], axis=0)
    source_centered = source - source_mean
    target_centered = target - target_mean

    covariance = (target_centered * weights[:, None]).T @ source_centered
    U, singular, Vt = np.linalg.svd(covariance)
    correction = np.eye(3)
    correction[-1, -1] = np.sign(np.linalg.det(U @ Vt))
    rotation = U @ correction @ Vt

    variance = np.sum(weights * np.sum(source_centered ** 2, axis=1))
    scale = float(np.sum(singular * np.diag(correction)) / max(variance, 1e-12))
    translation = target_mean - scale * (rotation @ source_mean)
    return scale, rotation, translation


def principal_axes(points):
    centered = points - np.mean(points, axis=0)
    covariance = centered.T @ centered / max(len(points) - 1, 1)
    values, vectors = np.linalg.eigh(covariance)
    vectors = vectors[:, np.argsort(values)[::-1]]
    if np.linalg.det(vectors) < 0:
        vectors[:, -1] *= -1
    return vectors


def proper_axis_maps():
    maps = []
    for perm in permutations(range(3)):
        P = np.eye(3)[:, perm]
        for signs in product((-1.0, 1.0), repeat=3):
            S = P @ np.diag(signs)
            if np.linalg.det(S) > 0.5:
                maps.append(S)
    return maps


def robust_radius(points):
    center = np.median(points, axis=0)
    return float(np.median(np.linalg.norm(points - center, axis=1)))


def partial_alignment_score(source_aligned, target, trim=0.68):
    target_tree = cKDTree(target)
    forward = target_tree.query(source_aligned, k=1, workers=-1)[0]
    cutoff_f = np.quantile(forward, trim)
    forward_score = forward[forward <= cutoff_f].mean()

    source_tree = cKDTree(source_aligned)
    reverse = source_tree.query(target, k=1, workers=-1)[0]
    cutoff_r = np.quantile(reverse, 0.90)
    reverse_score = reverse[reverse <= cutoff_r].mean()
    return float(forward_score + 0.15 * reverse_score)


def pca_similarity_initializer(source, target, score_sample=PCA_SCORE_SAMPLE):
    rng = np.random.default_rng(SEED)
    source_score = sample_rows(source, score_sample, rng)
    target_score = sample_rows(target, score_sample, rng)
    source_center = np.median(source_score, axis=0)
    target_center = np.median(target_score, axis=0)
    source_axes = principal_axes(source_score)
    target_axes = principal_axes(target_score)
    scale = robust_radius(target_score) / max(robust_radius(source_score), 1e-12)

    hypotheses = []
    for axis_map in proper_axis_maps():
        rotation = target_axes @ axis_map @ source_axes.T
        translation = target_center - scale * (rotation @ source_center)
        aligned = transform_points(source_score, scale, rotation, translation)
        score = partial_alignment_score(aligned, target_score)
        hypotheses.append((score, scale, rotation, translation))
    hypotheses.sort(key=lambda item: item[0])
    return hypotheses


def robust_partial_icp_sim3(
    source,
    target,
    source_normals,
    target_normals,
    initial,
    iterations=PARTIAL_ICP_ITERATIONS,
    trim=PARTIAL_ICP_TRIM_QUANTILE,
):
    scale, rotation, translation = initial
    initial_scale = float(scale)
    target_tree = cKDTree(target)
    history = []

    for iteration in range(iterations):
        aligned = transform_points(source, scale, rotation, translation)
        distances, indices = target_tree.query(aligned, k=1, workers=-1)
        cutoff = max(float(np.quantile(distances, trim)), 1e-12)
        keep = distances <= cutoff

        if source_normals is not None and target_normals is not None:
            aligned_normals = transform_normals(source_normals, rotation)
            cosine = np.abs(np.sum(aligned_normals * target_normals[indices], axis=1))
            normal_keep = cosine >= PARTIAL_ICP_NORMAL_COS
            if np.count_nonzero(keep & normal_keep) >= 500:
                keep &= normal_keep

        residual = distances[keep] / cutoff
        weights = np.square(np.clip(1.0 - residual ** 2, 0.0, None))
        if np.count_nonzero(weights > 0) < 100:
            raise RuntimeError("Too few partial-ICP correspondences. Supply manual landmarks.")

        new_scale, new_rotation, new_translation = weighted_umeyama(
            source[keep], target[indices[keep]], weights
        )
        new_scale = float(np.clip(
            new_scale,
            initial_scale * SIM3_SCALE_WINDOW[0],
            initial_scale * SIM3_SCALE_WINDOW[1],
        ))

        delta = (
            abs(math.log(max(new_scale, 1e-12) / max(scale, 1e-12)))
            + np.linalg.norm(new_translation - translation) / VGGT_DIAGONAL
            + np.linalg.norm(new_rotation - rotation)
        )
        scale, rotation, translation = new_scale, new_rotation, new_translation
        history.append({
            "iteration": iteration,
            "trimmed_mean": float(distances[keep].mean()),
            "cutoff": cutoff,
            "inliers": int(np.count_nonzero(keep)),
            "scale": scale,
            "delta": float(delta),
        })
        if delta < 1e-7:
            break

    return scale, rotation, translation, history


In [ ]:
rng = np.random.default_rng(SEED)
coarse_prior_points, coarse_prior_normals = sample_surface_with_normals(prior_mesh, COARSE_SAMPLE_PRIOR)

vggt_idx = rng.choice(
    len(vggt_points),
    min(COARSE_SAMPLE_VGGT, len(vggt_points)),
    replace=False,
)
coarse_vggt_points = vggt_points[vggt_idx]
coarse_vggt_normals = vggt_normals[vggt_idx]

if len(LANDMARKS_PRIOR) >= 3:
    initial_scale, initial_rotation, initial_translation = weighted_umeyama(
        LANDMARKS_PRIOR, LANDMARKS_VGGT
    )
    initialization_method = "manual_landmarks_umeyama"
    pca_hypothesis_report = []
else:
    hypotheses = pca_similarity_initializer(coarse_prior_points, coarse_vggt_points)
    if not 0 <= PCA_HYPOTHESIS_RANK < len(hypotheses):
        raise ValueError(f"PCA_HYPOTHESIS_RANK must be between 0 and {len(hypotheses) - 1}.")
    best_score, initial_scale, initial_rotation, initial_translation = hypotheses[PCA_HYPOTHESIS_RANK]
    initialization_method = "automatic_pca_24_hypotheses"
    pca_hypothesis_report = [
        {"rank": rank, "score": float(item[0]), "scale": float(item[1])}
        for rank, item in enumerate(hypotheses[:8])
    ]
    print("Best PCA hypotheses:")
    print(json.dumps(pca_hypothesis_report, indent=2))

coarse_scale, coarse_rotation, coarse_translation, icp_history = robust_partial_icp_sim3(
    coarse_prior_points,
    coarse_vggt_points,
    coarse_prior_normals,
    coarse_vggt_normals,
    (initial_scale, initial_rotation, initial_translation),
)

coarse_transform = sim3_matrix(coarse_scale, coarse_rotation, coarse_translation)
coarse_mesh = prior_mesh.copy()
coarse_mesh.apply_transform(coarse_transform)
coarse_mesh.export(OUT_DIR / "prior_coarse_sim3.glb")

coarse_preview, _ = sample_surface_with_normals(coarse_mesh, 60_000, seed=SEED + 1)
fig = overlay_figure(vggt_points, coarse_preview, title="Coarse partial Sim(3) — inspect before trusting refinement")
fig.show()
fig.write_html(OUT_DIR / "01_coarse_sim3_overlay.html", include_plotlyjs=True, full_html=True)

coarse_record = {
    "method": initialization_method,
    "scale": float(coarse_scale),
    "rotation": coarse_rotation.tolist(),
    "translation": coarse_translation.tolist(),
    "matrix": coarse_transform.tolist(),
    "manual_landmark_pairs": int(len(LANDMARKS_PRIOR)),
    "selected_pca_hypothesis_rank": int(PCA_HYPOTHESIS_RANK) if len(LANDMARKS_PRIOR) == 0 else None,
    "pca_hypotheses": pca_hypothesis_report,
    "icp_history": icp_history,
}
(OUT_DIR / "coarse_sim3.json").write_text(json.dumps(coarse_record, indent=2))
print("coarse scale:", coarse_scale)
print("coarse trimmed residual:", icp_history[-1]["trimmed_mean"])


## 5. Kaolin differentiable overlap refinement

The rigid refinement uses Kaolin's one-sided nearest-surface distance from the intact prior to the
measured cloud. A detached robust trim mask keeps the generated missing region from pulling the
transform. A low-weight reverse term prevents collapse onto only a tiny measured patch. Normal,
landmark, and initialization regularizers stabilize the fit.

This is deliberately **not** a symmetric whole-shape Chamfer loss.


In [ ]:
def axis_angle_matrix_torch(rotvec):
    """Differentiable Rodrigues map; accepts shape (..., 3)."""
    wx, wy, wz = rotvec.unbind(dim=-1)
    zeros = torch.zeros_like(wx)
    skew = torch.stack(
        [zeros, -wz, wy, wz, zeros, -wx, -wy, wx, zeros], dim=-1
    ).reshape(rotvec.shape[:-1] + (3, 3))
    theta = torch.linalg.norm(rotvec, dim=-1)
    A = torch.sinc(theta / math.pi)[..., None, None]
    B = 0.5 * torch.sinc(theta / (2.0 * math.pi))[..., None, None] ** 2
    eye = torch.eye(3, dtype=rotvec.dtype, device=rotvec.device)
    eye = eye.expand(rotvec.shape[:-1] + (3, 3))
    return eye + A * skew + B * (skew @ skew)


def kaolin_sided_distance(source, target):
    """Version-tolerant return order for Kaolin 0.18 and older releases."""
    first, second = kal.metrics.pointcloud.sided_distance(source, target)
    if first.dtype in (torch.int32, torch.int64):
        indices, distances_squared = first, second
    else:
        distances_squared, indices = first, second
    return distances_squared, indices.long()


def robust_trimmed_mean_distance(distances_squared, quantile):
    cutoff = torch.quantile(distances_squared.detach(), quantile).clamp_min(1e-12)
    normalized = distances_squared.detach() / cutoff
    weights = torch.clamp(1.0 - normalized, min=0.0) ** 2
    distance = torch.sqrt(distances_squared + 1e-12)
    mean = torch.sum(weights * distance) / torch.sum(weights).clamp_min(1.0)
    return mean, weights, cutoff


def optimize_kaolin_sim3(
    prior_points_np,
    prior_normals_np,
    measured_points_np,
    measured_normals_np,
    initial_sim3,
):
    initial_scale, initial_rotation, initial_translation = initial_sim3
    prior_points_t = torch.as_tensor(prior_points_np, dtype=torch.float32, device=DEVICE)[None]
    prior_normals_t = torch.as_tensor(prior_normals_np, dtype=torch.float32, device=DEVICE)[None]
    measured_points_t = torch.as_tensor(measured_points_np, dtype=torch.float32, device=DEVICE)[None]
    measured_normals_t = torch.as_tensor(measured_normals_np, dtype=torch.float32, device=DEVICE)[None]

    initial_rotvec = Rotation.from_matrix(initial_rotation).as_rotvec()
    log_scale = torch.nn.Parameter(torch.tensor(math.log(initial_scale), dtype=torch.float32, device=DEVICE))
    rotvec = torch.nn.Parameter(torch.tensor(initial_rotvec, dtype=torch.float32, device=DEVICE))
    translation = torch.nn.Parameter(torch.tensor(initial_translation, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.Adam([log_scale, rotvec, translation], lr=KAOLIN_RIGID_LR)

    log_scale_0 = log_scale.detach().clone()
    rotvec_0 = rotvec.detach().clone()
    translation_0 = translation.detach().clone()
    diag_t = torch.tensor(VGGT_DIAGONAL, dtype=torch.float32, device=DEVICE)

    if len(LANDMARKS_PRIOR):
        landmark_prior_t = torch.as_tensor(LANDMARKS_PRIOR, dtype=torch.float32, device=DEVICE)
        landmark_vggt_t = torch.as_tensor(LANDMARKS_VGGT, dtype=torch.float32, device=DEVICE)
    else:
        landmark_prior_t = landmark_vggt_t = None

    history = []
    for step in range(KAOLIN_RIGID_STEPS):
        optimizer.zero_grad(set_to_none=True)
        scale = torch.exp(log_scale)
        rotation = axis_angle_matrix_torch(rotvec)
        aligned = scale * (prior_points_t @ rotation.T) + translation

        forward_d2, forward_idx = kaolin_sided_distance(aligned, measured_points_t)
        forward_mean, forward_weights, _ = robust_trimmed_mean_distance(
            forward_d2, KAOLIN_TRIM_QUANTILE
        )
        loss_surface = forward_mean / diag_t

        reverse_d2, _ = kaolin_sided_distance(measured_points_t, aligned)
        reverse_mean, _, _ = robust_trimmed_mean_distance(reverse_d2, 0.90)
        loss_reverse = reverse_mean / diag_t

        aligned_normals = prior_normals_t @ rotation.T
        paired_normals = measured_normals_t[:, forward_idx[0], :]
        cosine = torch.abs(torch.sum(aligned_normals * paired_normals, dim=-1))
        normal_penalty = 1.0 - cosine.clamp(0.0, 1.0)
        loss_normal = torch.sum(forward_weights * normal_penalty) / torch.sum(forward_weights).clamp_min(1.0)

        if landmark_prior_t is not None:
            aligned_landmarks = scale * (landmark_prior_t @ rotation.T) + translation
            loss_landmark = torch.mean(torch.linalg.norm(aligned_landmarks - landmark_vggt_t, dim=1)) / diag_t
        else:
            loss_landmark = torch.zeros((), device=DEVICE)

        loss_reg = (
            (log_scale - log_scale_0) ** 2
            + torch.mean((rotvec - rotvec_0) ** 2)
            + torch.mean(((translation - translation_0) / diag_t) ** 2)
        )
        loss = (
            loss_surface
            + KAOLIN_REVERSE_WEIGHT * loss_reverse
            + KAOLIN_NORMAL_WEIGHT * loss_normal
            + KAOLIN_LANDMARK_WEIGHT * loss_landmark
            + KAOLIN_REG_WEIGHT * loss_reg
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_([log_scale, rotvec, translation], 5.0)
        optimizer.step()

        with torch.no_grad():
            lo = log_scale_0 + math.log(SIM3_SCALE_WINDOW[0])
            hi = log_scale_0 + math.log(SIM3_SCALE_WINDOW[1])
            log_scale.clamp_(lo, hi)

        if step % 25 == 0 or step == KAOLIN_RIGID_STEPS - 1:
            item = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "surface": float(loss_surface.detach().cpu()),
                "reverse": float(loss_reverse.detach().cpu()),
                "normal": float(loss_normal.detach().cpu()),
                "landmark": float(loss_landmark.detach().cpu()),
                "scale": float(torch.exp(log_scale).detach().cpu()),
            }
            history.append(item)
            print(item)

    final_scale = float(torch.exp(log_scale).detach().cpu())
    final_rotation = axis_angle_matrix_torch(rotvec).detach().cpu().numpy()
    final_translation = translation.detach().cpu().numpy()
    return final_scale, final_rotation, final_translation, history


In [ ]:
rng = np.random.default_rng(SEED + 2)
kaolin_prior_points, kaolin_prior_normals = sample_surface_with_normals(
    prior_mesh, KAOLIN_PRIOR_POINTS, seed=SEED + 2
)
measured_idx = rng.choice(
    len(vggt_points), min(KAOLIN_VGGT_POINTS, len(vggt_points)), replace=False
)

rigid_scale, rigid_rotation, rigid_translation, rigid_history = optimize_kaolin_sim3(
    kaolin_prior_points,
    kaolin_prior_normals,
    vggt_points[measured_idx],
    vggt_normals[measured_idx],
    (coarse_scale, coarse_rotation, coarse_translation),
)

rigid_transform = sim3_matrix(rigid_scale, rigid_rotation, rigid_translation)
rigid_mesh = prior_mesh.copy()
rigid_mesh.apply_transform(rigid_transform)
rigid_mesh.export(OUT_DIR / "prior_kaolin_aligned.glb")

rigid_preview, _ = sample_surface_with_normals(rigid_mesh, 60_000, seed=SEED + 3)
fig = overlay_figure(vggt_points, rigid_preview, title="Kaolin refined partial Sim(3)")
fig.show()
fig.write_html(OUT_DIR / "02_kaolin_rigid_overlay.html", include_plotlyjs=True, full_html=True)

rigid_record = {
    "parameterization": "Sim(3): uniform scale, SO(3), translation",
    "scale": rigid_scale,
    "rotation": rigid_rotation.tolist(),
    "translation": rigid_translation.tolist(),
    "matrix": rigid_transform.tolist(),
    "history": rigid_history,
}
(OUT_DIR / "similarity_transform.json").write_text(json.dumps(rigid_record, indent=2))


## 6. Generic constrained non-rigid adaptation

This replaces any hard-coded “thin the tail” operation. An embedded deformation graph places local
SE(3) transforms over the prior. Only prior vertices already near measured geometry contribute to
the data term. ARAP, anchor, rotation, translation, and displacement limits propagate a cautious,
smooth deformation into unsupported regions.

Set `RUN_GENERIC_DEFORMATION=False` when the rigid fit is already adequate or when the prior is too
uncertain to justify local adaptation.


In [ ]:
def measured_spacing(points, sample_count=12_000):
    rng = np.random.default_rng(SEED)
    sample = sample_rows(points, sample_count, rng)
    distances = cKDTree(points).query(sample, k=2, workers=-1)[0][:, 1]
    return float(np.median(distances[np.isfinite(distances)]))


MEASURED_SPACING = measured_spacing(vggt_points)
SUPPORT_DISTANCE = max(
    SUPPORT_REL_DIAGONAL * VGGT_DIAGONAL,
    SUPPORT_SPACING_MULTIPLIER * MEASURED_SPACING,
)
DEFORM_FIT_DISTANCE = DEFORM_FIT_REL_DIAGONAL * VGGT_DIAGONAL
print("measured spacing:", MEASURED_SPACING)
print("support distance:", SUPPORT_DISTANCE)
print("deformation fit distance:", DEFORM_FIT_DISTANCE)


def farthest_point_nodes(vertices, node_count, candidate_limit=60_000):
    rng = np.random.default_rng(SEED)
    vertices = np.asarray(vertices)
    if len(vertices) > candidate_limit:
        candidates = vertices[rng.choice(len(vertices), candidate_limit, replace=False)]
    else:
        candidates = vertices
    node_count = min(node_count, len(candidates))
    chosen = [int(np.argmax(np.linalg.norm(candidates - np.mean(candidates, axis=0), axis=1)))]
    min_d2 = np.sum((candidates - candidates[chosen[0]]) ** 2, axis=1)
    for _ in range(1, node_count):
        index = int(np.argmax(min_d2))
        chosen.append(index)
        min_d2 = np.minimum(min_d2, np.sum((candidates - candidates[index]) ** 2, axis=1))
    return candidates[np.asarray(chosen)]


def graph_knn_weights(points, nodes, k=4):
    distances, indices = cKDTree(nodes).query(points, k=min(k, len(nodes)), workers=-1)
    if distances.ndim == 1:
        distances = distances[:, None]
        indices = indices[:, None]
    sigma = np.maximum(distances[:, -1:], 1e-8)
    weights = np.exp(-0.5 * (distances / sigma) ** 2)
    weights /= np.maximum(weights.sum(axis=1, keepdims=True), 1e-12)
    return indices.astype(np.int64), weights.astype(np.float32)


def graph_edges(nodes, neighbors=5):
    _, indices = cKDTree(nodes).query(nodes, k=min(neighbors + 1, len(nodes)))
    edges = set()
    for i, row in enumerate(np.atleast_2d(indices)):
        for j in row[1:]:
            a, b = sorted((int(i), int(j)))
            if a != b:
                edges.add((a, b))
    return np.asarray(sorted(edges), dtype=np.int64)


def deform_points_torch(points, nodes, rotations, translations, node_ids, weights):
    local_nodes = nodes[node_ids]
    local_rotations = rotations[node_ids]
    local_translations = translations[node_ids]
    local = points[:, None, :] - local_nodes
    rotated = torch.matmul(local_rotations, local[..., None]).squeeze(-1)
    candidates = rotated + local_nodes + local_translations
    return torch.sum(weights[..., None] * candidates, dim=1)


def deform_normals_torch(normals, rotations, node_ids, weights):
    local_rotations = rotations[node_ids]
    rotated = torch.matmul(local_rotations, normals[:, None, :, None]).squeeze(-1)
    result = torch.sum(weights[..., None] * rotated, dim=1)
    return torch.nn.functional.normalize(result, dim=-1)


def clamp_row_norm_(tensor, maximum):
    with torch.no_grad():
        norms = torch.linalg.norm(tensor, dim=1, keepdim=True).clamp_min(1e-12)
        tensor.mul_(torch.clamp(maximum / norms, max=1.0))


def optimize_deformation_graph(mesh, measured_points, measured_normals):
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    vertex_normals = np.asarray(mesh.vertex_normals, dtype=np.float64)
    measured_tree = cKDTree(measured_points)
    distances, nearest = measured_tree.query(vertices, k=1, workers=-1)
    normal_cosine = np.abs(np.sum(vertex_normals * measured_normals[nearest], axis=1))

    fit_mask = (distances <= DEFORM_FIT_DISTANCE) & (normal_cosine >= math.cos(math.radians(80.0)))
    fit_indices = np.flatnonzero(fit_mask)
    if len(fit_indices) < 500:
        raise RuntimeError(
            "Too few measured-overlap vertices for non-rigid adaptation. "
            "Fix the rigid alignment or set RUN_GENERIC_DEFORMATION=False."
        )
    rng = np.random.default_rng(SEED + 4)
    if len(fit_indices) > 12_000:
        fit_indices = rng.choice(fit_indices, 12_000, replace=False)

    nodes_np = farthest_point_nodes(vertices, DEFORM_GRAPH_NODES)
    edges_np = graph_edges(nodes_np)
    fit_node_ids_np, fit_weights_np = graph_knn_weights(vertices[fit_indices], nodes_np, DEFORM_GRAPH_KNN)

    node_distances = measured_tree.query(nodes_np, k=1, workers=-1)[0]
    anchor_mask_np = node_distances <= SUPPORT_DISTANCE

    measured_idx = rng.choice(
        len(measured_points), min(KAOLIN_VGGT_POINTS, len(measured_points)), replace=False
    )
    target_points_t = torch.as_tensor(measured_points[measured_idx], dtype=torch.float32, device=DEVICE)[None]
    target_normals_t = torch.as_tensor(measured_normals[measured_idx], dtype=torch.float32, device=DEVICE)[None]

    fit_points_t = torch.as_tensor(vertices[fit_indices], dtype=torch.float32, device=DEVICE)
    fit_normals_t = torch.as_tensor(vertex_normals[fit_indices], dtype=torch.float32, device=DEVICE)
    nodes_t = torch.as_tensor(nodes_np, dtype=torch.float32, device=DEVICE)
    edges_t = torch.as_tensor(edges_np, dtype=torch.long, device=DEVICE)
    fit_node_ids_t = torch.as_tensor(fit_node_ids_np, dtype=torch.long, device=DEVICE)
    fit_weights_t = torch.as_tensor(fit_weights_np, dtype=torch.float32, device=DEVICE)
    anchor_mask_t = torch.as_tensor(anchor_mask_np, dtype=torch.bool, device=DEVICE)
    diag_t = torch.tensor(VGGT_DIAGONAL, dtype=torch.float32, device=DEVICE)

    node_rotvec = torch.nn.Parameter(torch.zeros((len(nodes_np), 3), dtype=torch.float32, device=DEVICE))
    node_translation = torch.nn.Parameter(torch.zeros((len(nodes_np), 3), dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.Adam([node_rotvec, node_translation], lr=DEFORM_LR)
    history = []

    for step in range(DEFORM_STEPS):
        optimizer.zero_grad(set_to_none=True)
        rotations = axis_angle_matrix_torch(node_rotvec)
        deformed_fit = deform_points_torch(
            fit_points_t, nodes_t, rotations, node_translation, fit_node_ids_t, fit_weights_t
        )
        deformed_normals = deform_normals_torch(
            fit_normals_t, rotations, fit_node_ids_t, fit_weights_t
        )

        distances2, nearest_idx = kaolin_sided_distance(deformed_fit[None], target_points_t)
        fit_mean, fit_weights_robust, _ = robust_trimmed_mean_distance(distances2, 0.90)
        loss_fit = fit_mean / diag_t

        paired_normals = target_normals_t[:, nearest_idx[0], :]
        cosine = torch.abs(torch.sum(deformed_normals[None] * paired_normals, dim=-1))
        loss_normal = torch.sum(fit_weights_robust * (1.0 - cosine)) / torch.sum(fit_weights_robust).clamp_min(1.0)

        i, j = edges_t[:, 0], edges_t[:, 1]
        edge_vector = nodes_t[j] - nodes_t[i]
        predicted_j = torch.matmul(rotations[i], edge_vector[..., None]).squeeze(-1) + nodes_t[i] + node_translation[i]
        actual_j = nodes_t[j] + node_translation[j]
        loss_arap = torch.mean(torch.sum((predicted_j - actual_j) ** 2, dim=1)) / (diag_t ** 2)

        if torch.any(anchor_mask_t):
            loss_anchor = torch.mean(torch.sum(node_translation[anchor_mask_t] ** 2, dim=1)) / (diag_t ** 2)
        else:
            loss_anchor = torch.zeros((), device=DEVICE)
        loss_rotation = torch.mean(node_rotvec ** 2)
        loss_translation = torch.mean(torch.sum(node_translation ** 2, dim=1)) / (diag_t ** 2)

        loss = (
            loss_fit
            + DEFORM_W_NORMAL * loss_normal
            + DEFORM_W_ARAP * loss_arap
            + DEFORM_W_ANCHOR * loss_anchor
            + DEFORM_W_ROTATION * loss_rotation
            + DEFORM_W_TRANSLATION * loss_translation
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_([node_rotvec, node_translation], 5.0)
        optimizer.step()
        clamp_row_norm_(node_translation, DEFORM_MAX_TRANSLATION_REL * VGGT_DIAGONAL)
        clamp_row_norm_(node_rotvec, math.radians(DEFORM_MAX_ROTATION_DEG))

        if step % 25 == 0 or step == DEFORM_STEPS - 1:
            item = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "fit": float(loss_fit.detach().cpu()),
                "normal": float(loss_normal.detach().cpu()),
                "arap": float(loss_arap.detach().cpu()),
                "anchor": float(loss_anchor.detach().cpu()),
            }
            history.append(item)
            print(item)

    rotations_np = axis_angle_matrix_torch(node_rotvec).detach().cpu().numpy()
    translations_np = node_translation.detach().cpu().numpy()

    all_node_ids, all_weights = graph_knn_weights(vertices, nodes_np, DEFORM_GRAPH_KNN)
    deformed_chunks = []
    chunk_size = 100_000
    with torch.no_grad():
        rotations_t = torch.as_tensor(rotations_np, dtype=torch.float32, device=DEVICE)
        translations_t = torch.as_tensor(translations_np, dtype=torch.float32, device=DEVICE)
        for start in range(0, len(vertices), chunk_size):
            stop = min(start + chunk_size, len(vertices))
            points_t = torch.as_tensor(vertices[start:stop], dtype=torch.float32, device=DEVICE)
            ids_t = torch.as_tensor(all_node_ids[start:stop], dtype=torch.long, device=DEVICE)
            weights_t = torch.as_tensor(all_weights[start:stop], dtype=torch.float32, device=DEVICE)
            deformed = deform_points_torch(points_t, nodes_t, rotations_t, translations_t, ids_t, weights_t)
            deformed_chunks.append(deformed.cpu().numpy())

    result = mesh.copy()
    result.vertices = np.concatenate(deformed_chunks, axis=0)
    result.remove_unreferenced_vertices()
    graph_state = {
        "nodes": nodes_np,
        "rotations": rotations_np,
        "translations": translations_np,
        "edges": edges_np,
        "fit_vertex_indices": fit_indices,
        "anchor_mask": anchor_mask_np,
        "history": history,
    }
    return result, graph_state


In [ ]:
if RUN_GENERIC_DEFORMATION:
    adapted_mesh, deformation_state = optimize_deformation_graph(
        rigid_mesh, vggt_points, vggt_normals
    )
    np.savez_compressed(
        OUT_DIR / "deformation_graph.npz",
        nodes=deformation_state["nodes"],
        rotations=deformation_state["rotations"],
        translations=deformation_state["translations"],
        edges=deformation_state["edges"],
        fit_vertex_indices=deformation_state["fit_vertex_indices"],
        anchor_mask=deformation_state["anchor_mask"],
    )
    (OUT_DIR / "deformation_history.json").write_text(
        json.dumps(deformation_state["history"], indent=2)
    )
else:
    adapted_mesh = rigid_mesh.copy()
    deformation_state = None

adapted_mesh.export(OUT_DIR / "prior_adapted.glb")
adapted_preview, _ = sample_surface_with_normals(adapted_mesh, 60_000, seed=SEED + 5)
fig = overlay_figure(vggt_points, adapted_preview, title="Adapted prior — VGGT remains unchanged")
fig.show()
fig.write_html(OUT_DIR / "03_adapted_prior_overlay.html", include_plotlyjs=True, full_html=True)


## 7. Supported / uncertain / unsupported classification

Classification uses metric proximity and normal agreement. It is intentionally conservative:
`unsupported` means “not supported by this point cloud,” not “certainly absent from the real
object.” Visibility-aware depth/silhouette evidence can later be added from VGGT camera outputs,
but the existing PLY contract alone cannot distinguish absence from occlusion.


In [ ]:
def classify_prior_vertices(mesh, measured_points, measured_normals):
    vertices = np.asarray(mesh.vertices)
    normals = np.asarray(mesh.vertex_normals)
    distances, nearest = cKDTree(measured_points).query(vertices, k=1, workers=-1)
    cosine = np.abs(np.sum(normals * measured_normals[nearest], axis=1))
    supported = (distances <= SUPPORT_DISTANCE) & (cosine >= SUPPORT_NORMAL_COS)
    uncertain = (~supported) & (distances <= UNCERTAIN_MULTIPLIER * SUPPORT_DISTANCE)
    state = np.full(len(vertices), 2, dtype=np.uint8)  # 0 supported, 1 uncertain, 2 unsupported
    state[supported] = 0
    state[uncertain] = 1
    return state, distances, cosine, nearest


vertex_state, support_distances, support_cosine, nearest_vggt = classify_prior_vertices(
    adapted_mesh, vggt_points, vggt_normals
)
state_names = np.array(["supported", "uncertain", "unsupported"])
counts = {state_names[i]: int(np.count_nonzero(vertex_state == i)) for i in range(3)}
fractions = {key: value / len(vertex_state) for key, value in counts.items()}
print("state counts:", counts)
print("state fractions:", fractions)

state_colors_rgb = np.array([
    [40, 190, 90, 255],    # supported — green
    [245, 165, 35, 255],   # uncertain — orange
    [225, 60, 70, 255],    # unsupported — red
], dtype=np.uint8)
state_mesh = adapted_mesh.copy()
state_mesh.visual.vertex_colors = state_colors_rgb[vertex_state]
state_mesh.export(OUT_DIR / "prior_support_classification.ply")

plotly_colors = np.array(["rgb(40,190,90)", "rgb(245,165,35)", "rgb(225,60,70)"])
fig = overlay_figure(
    vggt_points,
    np.asarray(adapted_mesh.vertices),
    prior_colors=plotly_colors[vertex_state],
    title="Prior support: green=supported, orange=uncertain, red=unsupported",
)
fig.show()
fig.write_html(OUT_DIR / "04_support_classification.html", include_plotlyjs=True, full_html=True)


## 8. Generic candidate components

Candidate faces must contain at least two unsupported vertices and no strongly supported vertex.
Connected components are ranked by surface area. Large components can still be unseen backsides;
inspect them in the interactive preview before confirming a missing piece.


In [ ]:
def build_candidate_components(mesh, vertex_state, min_faces=MIN_COMPONENT_FACES):
    face_states = vertex_state[np.asarray(mesh.faces)]
    candidate_face_mask = (np.sum(face_states == 2, axis=1) >= 2) & (np.sum(face_states == 0, axis=1) == 0)
    candidate_faces = np.flatnonzero(candidate_face_mask)
    if len(candidate_faces) == 0:
        return [], candidate_face_mask

    adjacency = np.asarray(mesh.face_adjacency)
    keep_edges = candidate_face_mask[adjacency[:, 0]] & candidate_face_mask[adjacency[:, 1]]
    adjacency_edges = adjacency[keep_edges]
    components = trimesh.graph.connected_components(
        adjacency_edges,
        nodes=candidate_faces,
        min_len=min_faces,
    )
    records = []
    for faces in components:
        faces = np.asarray(faces, dtype=np.int64)
        vertices = np.unique(np.asarray(mesh.faces)[faces].ravel())
        area = float(np.asarray(mesh.area_faces)[faces].sum())
        weights = np.asarray(mesh.area_faces)[faces]
        centers = np.asarray(mesh.triangles_center)[faces]
        centroid = np.average(centers, axis=0, weights=np.maximum(weights, 1e-12))
        records.append({
            "faces": faces,
            "vertices": vertices,
            "area": area,
            "centroid": centroid,
            "median_support_distance": float(np.median(support_distances[vertices])),
            "max_support_distance": float(np.max(support_distances[vertices])),
        })
    records.sort(key=lambda item: item["area"], reverse=True)
    for component_id, record in enumerate(records):
        record["component_id"] = component_id
    return records, candidate_face_mask


candidate_components, candidate_face_mask = build_candidate_components(adapted_mesh, vertex_state)
candidate_catalog = [
    {
        "component_id": item["component_id"],
        "faces": int(len(item["faces"])),
        "vertices": int(len(item["vertices"])),
        "area": item["area"],
        "centroid": item["centroid"].tolist(),
        "median_support_distance": item["median_support_distance"],
        "max_support_distance": item["max_support_distance"],
    }
    for item in candidate_components
]
(OUT_DIR / "missing_candidate_catalog.json").write_text(json.dumps(candidate_catalog, indent=2))
print(json.dumps(candidate_catalog[:MAX_COMPONENTS_TO_SHOW], indent=2))


def candidate_preview_figure(mesh, measured_points, components, max_components=MAX_COMPONENTS_TO_SHOW):
    rng = np.random.default_rng(SEED)
    measured_show = sample_rows(measured_points, 45_000, rng)
    traces = [
        go.Scatter3d(
            x=measured_show[:, 0], y=measured_show[:, 1], z=measured_show[:, 2],
            mode="markers", name="VGGT measured",
            marker=dict(size=1.5, color="rgb(130,135,140)", opacity=0.45),
        )
    ]
    palette = [
        "#ffd92f", "#66c2a5", "#fc8d62", "#8da0cb", "#e78ac3", "#a6d854",
        "#e5c494", "#b3b3b3", "#1f78b4", "#33a02c", "#e31a1c", "#6a3d9a",
    ]
    for item in components[:max_components]:
        component = mesh.submesh([item["faces"]], append=True, repair=False)
        traces.append(go.Mesh3d(
            x=component.vertices[:, 0], y=component.vertices[:, 1], z=component.vertices[:, 2],
            i=component.faces[:, 0], j=component.faces[:, 1], k=component.faces[:, 2],
            name=f"candidate {item['component_id']}",
            color=palette[item["component_id"] % len(palette)], opacity=0.85,
            flatshading=True,
        ))
    fig = go.Figure(traces)
    fig.update_layout(
        title="Unsupported candidate components — IDs match the catalog",
        scene=dict(aspectmode="data"), width=1000, height=780,
        margin=dict(l=0, r=0, b=0, t=45),
    )
    return fig


candidate_fig = candidate_preview_figure(adapted_mesh, vggt_points, candidate_components)
candidate_fig.show()
candidate_fig.write_html(OUT_DIR / "05_missing_candidates.html", include_plotlyjs=True, full_html=True)


## 9. User-confirmed extraction

Return to the configuration cell and either:

- set `SELECT_MISSING_COMPONENTS=[id, ...]`, or
- enter one or more `MISSING_SEEDS` in the aligned VGGT frame.

Then set `CONFIRM_OBSERVED_ABSENCE=True` only after checking the candidate against photographs or
depth/silhouette evidence. `exterior_patch` preserves the prior's open exterior surface. The
best-effort `solid_cap` option asks Trimesh to close simple holes, but it is not a replacement for
the measured fracture interface. A manufacturing-ready solid or hollow repair still requires the
registered fracture close-up and a complementary mating surface.


In [ ]:
def components_from_seeds(components, mesh, seeds):
    selected = set()
    if len(seeds) == 0:
        return selected
    seeds = np.asarray(seeds, dtype=np.float64).reshape(-1, 3)
    vertices = np.asarray(mesh.vertices)
    for seed in seeds:
        best = None
        for item in components:
            component_vertices = item["vertices"]
            local_distances = np.linalg.norm(vertices[component_vertices] - seed, axis=1)
            distance = float(local_distances.min())
            if best is None or distance < best[0]:
                best = (distance, item["component_id"])
        if best is not None:
            selected.add(best[1])
    return selected


selected_ids = set(int(value) for value in SELECT_MISSING_COMPONENTS)
selected_ids |= components_from_seeds(candidate_components, adapted_mesh, MISSING_SEEDS)
available_ids = {item["component_id"] for item in candidate_components}
unknown_ids = selected_ids - available_ids
if unknown_ids:
    raise ValueError(f"Unknown component IDs: {sorted(unknown_ids)}")

missing_piece = None
missing_piece_path = None
missing_piece_warnings = []

if not CONFIRM_OBSERVED_ABSENCE:
    print("Extraction is paused safely. Inspect 05_missing_candidates.html, select IDs, and set CONFIRM_OBSERVED_ABSENCE=True.")
elif not selected_ids:
    print("Extraction is paused: select at least one candidate component or add a missing-region seed.")
else:
    selected_faces = np.concatenate([
        candidate_components[component_id]["faces"] for component_id in sorted(selected_ids)
    ])
    missing_piece = adapted_mesh.submesh([selected_faces], append=True, repair=False)
    missing_piece.remove_unreferenced_vertices()

    if ATTACHMENT_MODE == "solid_cap":
        trimesh.repair.fix_normals(missing_piece, multibody=True)
        trimesh.repair.fill_holes(missing_piece)
        if not missing_piece.is_watertight:
            missing_piece_warnings.append(
                "Best-effort solid_cap remained open. Do not manufacture it; construct the measured mating surface."
            )
    elif ATTACHMENT_MODE != "exterior_patch":
        raise ValueError("ATTACHMENT_MODE must be 'exterior_patch' or 'solid_cap'.")

    missing_piece_path = OUT_DIR / "missing_piece_hypothesis.glb"
    missing_piece.export(missing_piece_path)
    missing_piece.export(OUT_DIR / "missing_piece_hypothesis.ply")
    if missing_piece.is_watertight:
        missing_piece.export(OUT_DIR / "missing_piece_hypothesis.stl")
    else:
        missing_piece_warnings.append(
            "The extracted exterior hypothesis is not watertight; STL was intentionally not exported."
        )

    print("selected components:", sorted(selected_ids))
    print("missing-piece vertices/faces:", len(missing_piece.vertices), len(missing_piece.faces))
    print("watertight:", missing_piece.is_watertight)
    print("saved:", missing_piece_path)
    for warning in missing_piece_warnings:
        print("WARNING:", warning)


## 10. Validation, overlay, and handoff report

Residuals are reported in both VGGT units and as a fraction of the measured bounding-box diagonal.
The completed visualization overlays the authoritative measured cloud with the inferred piece; it
does not alter the VGGT cloud.


In [ ]:
def alignment_residuals(mesh, measured_points, sample_count=40_000):
    sample, _ = sample_surface_with_normals(mesh, sample_count, seed=SEED + 9)
    distances = cKDTree(measured_points).query(sample, k=1, workers=-1)[0]
    trimmed = distances[distances <= np.quantile(distances, KAOLIN_TRIM_QUANTILE)]
    return {
        "all_q50": float(np.quantile(distances, 0.50)),
        "all_q90": float(np.quantile(distances, 0.90)),
        "trimmed_mean": float(trimmed.mean()),
        "trimmed_rmse": float(np.sqrt(np.mean(trimmed ** 2))),
        "trimmed_mean_rel_diagonal": float(trimmed.mean() / VGGT_DIAGONAL),
    }


def mesh_health(mesh):
    return {
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
        "watertight": bool(mesh.is_watertight),
        "winding_consistent": bool(mesh.is_winding_consistent),
        "euler_number": int(mesh.euler_number),
        "area": float(mesh.area),
        "volume": float(mesh.volume) if mesh.is_watertight else None,
        "bounds": np.asarray(mesh.bounds).tolist(),
    }


residual_report = alignment_residuals(adapted_mesh, vggt_points)
quality_warnings = []
if residual_report["trimmed_mean_rel_diagonal"] > 0.02:
    quality_warnings.append(
        "Trimmed mean residual exceeds 2% of the VGGT diagonal; review landmarks and coarse orientation."
    )
if len(LANDMARKS_PRIOR) == 0:
    quality_warnings.append(
        "Automatic PCA initialization was used. Symmetric or articulated objects should be checked with manual landmarks."
    )
quality_warnings.extend(missing_piece_warnings)

if missing_piece is not None:
    piece_surface, _ = sample_surface_with_normals(missing_piece, min(45_000, max(5_000, len(missing_piece.faces) * 3)), seed=SEED + 10)
    completed_fig = overlay_figure(
        vggt_points,
        piece_surface,
        prior_colors=np.full(len(piece_surface), "rgb(255,215,40)"),
        title="FUSE completion hypothesis — grey measured, yellow inferred",
    )
    completed_fig.show()
    completed_fig.write_html(OUT_DIR / "06_completed_overlay.html", include_plotlyjs=True, full_html=True)

report = {
    "stage": "alignment_and_missing_piece_hypothesis",
    "run_name": RUN_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "authority_rule": "VGGT measured geometry is fixed; only the intact prior moves/deforms.",
    "inputs": {
        "vggt_cloud": str(VGGT_CLOUD_PATH),
        "vggt_sha256": file_sha256(VGGT_CLOUD_PATH),
        "hunyuan_prior": str(HUNYUAN_PRIOR_PATH),
        "hunyuan_sha256": file_sha256(HUNYUAN_PRIOR_PATH),
        "hunyuan_manifest": str(HUNYUAN_MANIFEST_PATH) if HUNYUAN_MANIFEST_PATH else None,
    },
    "compatibility": {
        "vggt_contract": "cleaned point cloud PLY; normals preferred and estimated if absent",
        "hunyuan_contract": "selected intact prior GLB plus optional prior_manifest.json",
        "units": "VGGT coordinate units; not assumed to be millimetres",
    },
    "coarse_initialization": initialization_method,
    "manual_landmark_pairs": int(len(LANDMARKS_PRIOR)),
    "sim3": rigid_record,
    "generic_deformation_enabled": bool(RUN_GENERIC_DEFORMATION),
    "support": {
        "measured_spacing": MEASURED_SPACING,
        "support_distance": SUPPORT_DISTANCE,
        "counts": counts,
        "fractions": fractions,
    },
    "alignment_residuals": residual_report,
    "candidate_components": candidate_catalog,
    "selected_missing_components": sorted(selected_ids),
    "absence_confirmed": bool(CONFIRM_OBSERVED_ABSENCE),
    "attachment_mode": ATTACHMENT_MODE,
    "missing_piece": mesh_health(missing_piece) if missing_piece is not None else None,
    "warnings": quality_warnings,
    "outputs": {
        "coarse_prior": str(OUT_DIR / "prior_coarse_sim3.glb"),
        "rigid_prior": str(OUT_DIR / "prior_kaolin_aligned.glb"),
        "adapted_prior": str(OUT_DIR / "prior_adapted.glb"),
        "support_classification": str(OUT_DIR / "prior_support_classification.ply"),
        "candidate_preview": str(OUT_DIR / "05_missing_candidates.html"),
        "missing_piece": str(missing_piece_path) if missing_piece_path else None,
        "completed_overlay": str(OUT_DIR / "06_completed_overlay.html") if missing_piece is not None else None,
    },
    "next_stage_contract": {
        "measured_geometry": "unchanged VGGT cloud",
        "exterior_hypothesis": "missing_piece_hypothesis.glb when absence_confirmed",
        "manufacturing_gate": "register fracture close-up and build a complementary mating surface",
    },
}

(OUT_DIR / "alignment_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps({
    "residuals": residual_report,
    "warnings": quality_warnings,
    "report": str(OUT_DIR / "alignment_report.json"),
}, indent=2))


In [ ]:
print("\nStage 3 outputs:")
for path in sorted(OUT_DIR.iterdir()):
    print(" ", path.name)

required = [
    OUT_DIR / "input_contract.json",
    OUT_DIR / "coarse_sim3.json",
    OUT_DIR / "similarity_transform.json",
    OUT_DIR / "prior_kaolin_aligned.glb",
    OUT_DIR / "prior_adapted.glb",
    OUT_DIR / "prior_support_classification.ply",
    OUT_DIR / "missing_candidate_catalog.json",
    OUT_DIR / "05_missing_candidates.html",
    OUT_DIR / "alignment_report.json",
]
missing_required = [str(path) for path in required if not path.exists()]
if missing_required:
    raise RuntimeError("Missing required Stage 3 outputs:\n" + "\n".join(missing_required))

if not CONFIRM_OBSERVED_ABSENCE:
    print("\nAlignment is complete. Missing-piece export remains intentionally gated by user confirmation.")
elif missing_piece is None:
    print("\nAbsence was confirmed, but no component was selected; no missing piece was exported.")
else:
    print("\nMissing-piece hypothesis exported. Check alignment_report.json before any manufacturing step.")
